# makemore：从零手搓字符级语言模型

**makemore** 是 Andrej Karpathy 的经典教学项目：给它一堆名字，它就能"再造"更多以假乱真的新名字。
它的核心思想和 GPT 这类大语言模型完全一致——**根据已经出现的字符，预测下一个字符**，只不过这里的"词"是单个字母。

本笔记本用**同一份人名数据**，循序渐进地实现 3 个模型，越往后越接近真实生产中的神经网络：

| 模型 | 思路 | 关键点 |
|------|------|--------|
| ① Bigram（统计版） | 数一数"某字符后面接某字符"出现了几次 | 纯计数，无需训练 |
| ② Bigram（神经网络版） | 用一个 27×27 的权重矩阵 + 梯度下降 学出同样的规律 | 引入 one-hot、softmax、反向传播 |
| ③ MLP 多层感知机 | 用**前 3 个字符**做上下文，嵌入 + 隐藏层预测下一个字符 | 引入词嵌入、mini-batch、训练/验证集、交叉熵 |

> 💡 **每一行代码都有中文注释**，跟着从上往下运行即可。
>
> 📦 **数据**：同目录下的 `names.txt`（本仓库已附带一份小样本）。想要更强的效果，可以下载 Karpathy 的 32000 个名字全量数据：
> `https://raw.githubusercontent.com/karpathy/makemore/master/names.txt`，替换掉 `names.txt` 再重跑即可。


## 0. 导入依赖

In [ ]:
import torch                     # PyTorch：张量运算 + 自动求导（深度学习核心库）
import torch.nn.functional as F  # 常用函数集合：one_hot、cross_entropy、softmax 等
import matplotlib.pyplot as plt  # 画图库，用来可视化 bigram 计数矩阵
%matplotlib inline               # 让图像直接嵌在 notebook 里显示

## 1. 读取数据

把 `names.txt` 按行读进来，每一行就是一个名字。

In [ ]:
words = open('names.txt', 'r').read().splitlines()  # 读入文件并按换行切分成列表，每个元素是一个名字

print(f'名字总数: {len(words)}')      # 一共有多少个名字
print('前 10 个:', words[:10])        # 瞄一眼前 10 个名字长什么样
print('最短长度:', min(len(w) for w in words))  # 最短的名字有几个字母
print('最长长度:', max(len(w) for w in words))  # 最长的名字有几个字母

## 2. 建立字符 ↔ 数字 的映射（词表）

模型只认识数字，不认识字母，所以要给每个字符编号。
额外引入一个特殊符号 `'.'` 表示**名字的开头和结尾**（编号为 0），这样模型才知道一个名字何时开始、何时结束。

In [ ]:
chars = sorted(list(set(''.join(words))))       # 把所有名字拼成一个大字符串，去重后排序 -> ['a','b',...,'z']
stoi = {s: i + 1 for i, s in enumerate(chars)}  # string->int：字母映射到 1~26（0 留给特殊符号）
stoi['.'] = 0                                    # '.' 作为开始/结束标记，编号为 0
itos = {i: s for s, i in stoi.items()}          # int->string：反向映射，采样时把数字翻译回字母
vocab_size = len(itos)                          # 词表大小 = 26 个字母 + 1 个 '.' = 27

print('词表大小:', vocab_size)                   # 打印 27
print(itos)                                     # 打印完整的 数字->字符 映射表

## 3. 模型①：Bigram 统计版

**Bigram** = 只看**前 1 个字符**来预测下 1 个字符。
最朴素的做法：遍历所有名字，数一数"字符 i 后面紧跟字符 j"出现了多少次，存进一个 27×27 的计数矩阵 `N`。

In [ ]:
N = torch.zeros((27, 27), dtype=torch.int32)  # 27x27 计数矩阵，N[i, j] 表示 '字符 i 后面接字符 j' 的次数

for w in words:                        # 遍历每一个名字
    chs = ['.'] + list(w) + ['.']      # 首尾加 '.'，例如 'emma' -> ['.','e','m','m','a','.']
    for ch1, ch2 in zip(chs, chs[1:]): # 取相邻字符对(bigram)：(.,e)(e,m)(m,m)(m,a)(a,.)
        ix1 = stoi[ch1]                # 前一个字符的编号
        ix2 = stoi[ch2]                # 后一个字符的编号
        N[ix1, ix2] += 1               # 对应格子计数 +1

print('矩阵形状:', N.shape)             # torch.Size([27, 27])

可视化这个计数矩阵，直观看看哪些字符组合最常见（颜色越深次数越多）：

In [ ]:
plt.figure(figsize=(16, 16))                    # 设置一张大画布
plt.imshow(N, cmap='Blues')                     # 用蓝色深浅表示计数大小
for i in range(27):                             # 遍历每一行
    for j in range(27):                         # 遍历每一列
        chstr = itos[i] + itos[j]               # 该格子对应的字符对，如 'em'
        plt.text(j, i, chstr, ha='center', va='bottom', color='gray')      # 格子上方写字符对
        plt.text(j, i, N[i, j].item(), ha='center', va='top', color='gray')# 格子下方写出现次数
plt.axis('off')                                 # 隐藏坐标轴
plt.show()                                      # 显示图像

### 把"计数"变成"概率"

把每一行的计数除以该行总和，就得到概率分布：`P[i, j]` = 在字符 i 之后，下一个是字符 j 的概率。

- `+1` 是**拉普拉斯平滑**：避免某些组合次数为 0，否则后面取 `log` 会得到负无穷。
- `keepdim=True` 保证除法时**按行广播**正确（27×27 除以 27×1，而不是 27×27 除以 1×27）。

In [ ]:
P = (N + 1).float()          # 转成浮点数；+1 做平滑，防止出现概率为 0 的组合
P /= P.sum(1, keepdim=True)  # 每行各元素除以该行之和 -> 每一行变成一个概率分布(加起来=1)

print('第 0 行(即 . 之后)概率之和:', P[0].sum().item())  # 应该约等于 1.0

### 用统计模型生成名字

从 `'.'`（编号 0）开始，按当前字符的概率分布随机抽下一个字符，直到抽到 `'.'` 结束。

In [ ]:
g = torch.Generator().manual_seed(2147483647)  # 固定随机种子，保证每次运行结果一致（可复现）

for i in range(10):            # 生成 10 个名字
    out = []                   # 存放当前名字的字符
    ix = 0                     # 从 '.'(编号 0) 开始
    while True:                # 循环采样，直到遇到结束符
        p = P[ix]              # 取出'当前字符'这一行的概率分布
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率抽 1 个字符
        out.append(itos[ix])   # 把抽到的字符翻译回字母并记录
        if ix == 0:            # 如果抽到 '.'(结束符)
            break              # 结束当前名字
    print(''.join(out))        # 拼接并打印(结尾会带一个 '.')

### 给模型打分：损失函数（平均负对数似然）

怎么衡量模型好坏？看它对**真实数据**给出的概率有多高。
把每个真实 bigram 的概率取 `log` 再累加、取负、求平均，得到 **平均负对数似然 (loss)**：**越小越好**。

In [ ]:
log_likelihood = 0.0                   # 累加对数似然
n = 0                                  # 统计一共有多少个 bigram

for w in words:                        # 遍历每个名字
    chs = ['.'] + list(w) + ['.']      # 同样首尾加 '.'
    for ch1, ch2 in zip(chs, chs[1:]): # 遍历每个相邻字符对
        ix1 = stoi[ch1]                # 前字符编号
        ix2 = stoi[ch2]                # 后字符编号
        prob = P[ix1, ix2]             # 模型给这个真实 bigram 的概率
        log_likelihood += torch.log(prob)  # 取对数后累加
        n += 1                         # 计数 +1

nll = -log_likelihood                  # 负对数似然(取负号，因为我们要最小化)
print(f'平均负对数似然(loss): {nll / n:.4f}')  # 除以总数得到平均 loss，这是评判标准

## 4. 模型②：Bigram 神经网络版

同样是 bigram，但这次不"数数"，而是用**一个神经网络 + 梯度下降**把规律**学**出来。
效果会和统计版几乎一样，但这套"前向传播 → 算 loss → 反向传播 → 更新参数"的流程，正是所有深度学习模型的通用套路。

### 4.1 构造训练样本

把每个 bigram 拆成 (输入字符 `x`, 目标字符 `y`)。

In [ ]:
xs, ys = [], []                        # xs=输入字符编号列表, ys=目标(下一个)字符编号列表

for w in words:                        # 遍历每个名字
    chs = ['.'] + list(w) + ['.']      # 首尾加 '.'
    for ch1, ch2 in zip(chs, chs[1:]): # 遍历每个 bigram
        xs.append(stoi[ch1])           # 输入：当前字符
        ys.append(stoi[ch2])           # 目标：下一个字符

xs = torch.tensor(xs)                  # 转成张量
ys = torch.tensor(ys)                  # 转成张量
num = xs.nelement()                    # 训练样本总数
print('训练样本数:', num)

### 4.2 初始化权重

`W` 是 27×27 的权重矩阵，就是这个网络**唯一要学习的参数**。
`requires_grad=True` 告诉 PyTorch：请帮我追踪它、给它算梯度。

In [ ]:
g = torch.Generator().manual_seed(2147483647)                # 固定种子
W = torch.randn((27, 27), generator=g, requires_grad=True)   # 随机初始化 27x27 权重，需要求梯度

### 4.3 训练：梯度下降

核心四步循环：
1. **前向传播**：one-hot 输入 → 乘 `W` 得到 logits → `exp` → 归一化，得到概率（这三步合起来就是 **softmax**）。
2. **算 loss**：真实目标字符的概率取负对数、求平均，再加一点 L2 正则（相当于平滑）。
3. **反向传播**：`loss.backward()` 自动算出每个参数的梯度。
4. **更新参数**：沿梯度反方向走一小步。

In [ ]:
for k in range(200):                                  # 训练 200 轮
    # --- 前向传播 ---
    xenc = F.one_hot(xs, num_classes=27).float()      # 把输入编号变成 one-hot 向量 (num, 27)
    logits = xenc @ W                                 # 矩阵乘法，得到每个字符的'得分'(等价于 log 计数) (num, 27)
    counts = logits.exp()                             # 指数化 -> 全为正，相当于'计数'
    probs = counts / counts.sum(1, keepdim=True)      # 按行归一化成概率(exp+归一化 = softmax)
    # --- 计算损失 ---
    loss = -probs[torch.arange(num), ys].log().mean() + 0.01 * (W ** 2).mean()  # 平均负对数似然 + L2 正则
    # --- 反向传播 ---
    W.grad = None                                     # 清空上一轮残留的梯度
    loss.backward()                                   # 自动求导，算出 W 的梯度
    # --- 更新参数 ---
    W.data += -50 * W.grad                            # 沿梯度反方向更新(学习率 50)

    if k % 20 == 0:                                   # 每 20 轮打印一次
        print(f'第 {k:3d} 轮  loss = {loss.item():.4f}')  # 观察 loss 是否在下降

### 4.4 用神经网络生成名字

流程和统计版一样，只是概率改由网络前向计算得出。生成结果应与统计版非常接近。

In [ ]:
g = torch.Generator().manual_seed(2147483647)          # 固定种子

for i in range(10):                                    # 生成 10 个名字
    out = []                                           # 存字符
    ix = 0                                             # 从 '.' 开始
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=27).float()  # 当前字符转 one-hot
        logits = xenc @ W                              # 前向：得分
        counts = logits.exp()                          # 指数化
        p = counts / counts.sum(1, keepdim=True)       # softmax 得到概率分布
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # 按概率采样
        out.append(itos[ix])                           # 记录字符
        if ix == 0:                                    # 遇到结束符
            break
    print(''.join(out))                                # 打印名字

## 5. 模型③：MLP 多层感知机（更接近真实生产）

Bigram 的硬伤：只看**前 1 个字符**，上下文太短。
本节实现 Karpathy 复现的 **Bengio 2003** 经典结构：用**前 3 个字符**作为上下文，先把字符**嵌入(embedding)**成向量，再经过一个隐藏层预测下一个字符。

这里出现的所有概念——词嵌入、隐藏层、mini-batch 小批量训练、训练/验证/测试集划分、交叉熵损失、学习率衰减——都是**当今真实深度学习项目每天在用**的东西。

### 5.1 构造数据集（含滑动窗口 + 三划分）

`block_size = 3` 表示上下文长度。用一个"滑动窗口"把名字切成 (前 3 个字符 → 第 4 个字符) 的样本。
同时把数据按 8:1:1 划分为**训练/验证/测试**集：训练集用来学，验证集用来调参&判断过拟合，测试集最后才看。

In [ ]:
block_size = 3   # 上下文长度：用前 3 个字符预测下一个字符

def build_dataset(words):                 # 把名字列表转成 (X, Y) 训练数据
    X, Y = [], []                         # X=上下文(每个是 3 个编号), Y=目标字符编号
    for w in words:                       # 遍历名字
        context = [0] * block_size        # 初始上下文全是 '.'(编号 0)，即 [0, 0, 0]
        for ch in w + '.':               # 遍历名字每个字符(末尾补 '.' 作为结束目标)
            ix = stoi[ch]                 # 当前字符编号
            X.append(context)             # 记录当前上下文
            Y.append(ix)                  # 记录要预测的目标字符
            context = context[1:] + [ix]  # 滑动窗口：丢掉最旧字符，把当前字符接到末尾
    X = torch.tensor(X)                   # (样本数, block_size)
    Y = torch.tensor(Y)                   # (样本数,)
    return X, Y

import random                             # 用于打乱名字顺序
random.seed(42)                           # 固定随机种子
random.shuffle(words)                     # 原地打乱名字列表
n1 = int(0.8 * len(words))                # 前 80% 作训练集
n2 = int(0.9 * len(words))                # 80%~90% 作验证集，最后 10% 作测试集

Xtr,  Ytr  = build_dataset(words[:n1])    # 训练集(train)
Xdev, Ydev = build_dataset(words[n1:n2])  # 验证集(dev/validation)
Xte,  Yte  = build_dataset(words[n2:])    # 测试集(test)
print('训练集:', Xtr.shape, Ytr.shape)     # 看形状：(样本数, 3) 和 (样本数,)

### 5.2 初始化网络参数

- `C`：**嵌入表**，把 27 个字符各自映射成一个 10 维向量（可学习）。
- `W1, b1`：隐藏层（输入 3×10=30 维 → 200 个神经元）。
- `W2, b2`：输出层（200 → 27 个字符的得分）。

In [ ]:
g = torch.Generator().manual_seed(2147483647)   # 固定种子
n_embd   = 10                                    # 每个字符嵌入成 10 维向量
n_hidden = 200                                   # 隐藏层神经元个数

C  = torch.randn((27, n_embd),               generator=g)  # 嵌入表：27 个字符 x 10 维
W1 = torch.randn((n_embd * block_size, n_hidden), generator=g)  # 隐藏层权重：30 -> 200
b1 = torch.randn(n_hidden,                   generator=g)  # 隐藏层偏置
W2 = torch.randn((n_hidden, 27),             generator=g)  # 输出层权重：200 -> 27
b2 = torch.randn(27,                         generator=g)  # 输出层偏置

parameters = [C, W1, b1, W2, b2]                            # 把所有参数收集到一个列表
print('参数总量:', sum(p.nelement() for p in parameters))   # 打印可训练参数个数
for p in parameters:                                        # 遍历每个参数张量
    p.requires_grad = True                                  # 标记：需要为它计算梯度

### 5.3 训练（mini-batch + 学习率衰减）

真实项目里数据量大，不会每步都用全部数据，而是每次随机取一小批(**mini-batch**)，速度快很多。
另外后期把学习率调小(**衰减**)，让模型在最优点附近更精细地收敛。

In [ ]:
for i in range(20000):                                    # 训练 20000 步
    # --- 取一个 mini-batch(32 个样本) ---
    ix = torch.randint(0, Xtr.shape[0], (32,), generator=g)      # 随机抽 32 个样本下标
    Xb, Yb = Xtr[ix], Ytr[ix]                                    # 这一批的输入和目标
    # --- 前向传播 ---
    emb = C[Xb]                                                  # 查嵌入表 -> (32, 3, 10)
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)  # 展平成(32,30)过隐藏层+tanh激活 -> (32,200)
    logits = h @ W2 + b2                                         # 输出层得分 -> (32, 27)
    loss = F.cross_entropy(logits, Yb)                          # 交叉熵损失(内部含 softmax+NLL，数值更稳定)
    # --- 反向传播 ---
    for p in parameters:                                        # 遍历参数
        p.grad = None                                           # 清空梯度
    loss.backward()                                             # 自动求导
    # --- 更新参数 ---
    lr = 0.1 if i < 10000 else 0.01                             # 前 1 万步学习率 0.1，之后降到 0.01(衰减)
    for p in parameters:                                        # 遍历参数
        p.data += -lr * p.grad                                  # 梯度下降更新

    if i % 2000 == 0:                                           # 每 2000 步打印
        print(f'第 {i:5d} 步  loss = {loss.item():.4f}')

### 5.4 在训练集/验证集上评估

`@torch.no_grad()` 表示评估时不用追踪梯度(省内存、更快)。
如果**训练集 loss 明显低于验证集 loss**，说明模型**过拟合**了。

In [ ]:
@torch.no_grad()                                            # 装饰器：此函数内不计算梯度
def split_loss(X, Y):                                       # 计算某个数据集上的整体 loss
    emb = C[X]                                              # 嵌入
    h = torch.tanh(emb.view(-1, n_embd * block_size) @ W1 + b1)  # 隐藏层
    logits = h @ W2 + b2                                    # 输出层
    return F.cross_entropy(logits, Y).item()                # 交叉熵 loss

print('训练集 loss:', round(split_loss(Xtr,  Ytr),  4))      # 训练集表现
print('验证集 loss:', round(split_loss(Xdev, Ydev), 4))      # 验证集表现(判断是否过拟合)

### 5.5 用 MLP 生成名字

每一步用当前的 3 个字符做上下文，预测并采样下一个字符，再滑动窗口继续。

In [ ]:
g = torch.Generator().manual_seed(2147483647 + 10)          # 固定种子(换个数字得到不同结果)

for _ in range(15):                                         # 生成 15 个名字
    out = []                                               # 存字符编号
    context = [0] * block_size                             # 初始上下文全是 '.'
    while True:
        emb = C[torch.tensor([context])]                   # 当前上下文的嵌入 -> (1, 3, 10)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)          # 隐藏层
        logits = h @ W2 + b2                               # 输出得分
        probs = F.softmax(logits, dim=1)                   # softmax 转成概率分布
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()  # 按概率采样下一个字符
        context = context[1:] + [ix]                       # 滑动窗口：更新上下文
        out.append(ix)                                     # 记录
        if ix == 0:                                        # 遇到结束符 '.'
            break
    print(''.join(itos[i] for i in out))                   # 把编号翻译回字母并打印

## 6. 小结

| 模型 | 大概 loss（小样本数据下） | 特点 |
|------|--------------------------|------|
| ① Bigram 统计版 | ~2.4 | 无需训练，纯计数 |
| ② Bigram 神经网络版 | ~2.4 | 学出与统计版几乎相同的规律，跑通了训练全流程 |
| ③ MLP 多层感知机 | 更低 | 上下文更长、有词嵌入，生成的名字更像真名 |

> loss 的具体数值取决于数据量。用 32000 个名字的**全量数据**会明显更低、生成质量更好。

**你已经掌握的通用套路（每个深度学习项目都一样）：**
1. 数据 → 编号（词表 stoi / itos）
2. 构造 (输入 X, 目标 Y) 样本，并划分训练/验证/测试集
3. 前向传播 → 交叉熵算 loss → `backward()` 反向传播 → 梯度下降更新参数
4. 在验证集上评估、判断过拟合
5. 从模型里采样、生成新内容

**接着可以怎么玩：**
- 换用全量 `names.txt`，把 loss 压得更低。
- 调大 `block_size`（看更长上下文）、`n_embd`、`n_hidden`。
- 把这套结构继续升级为 RNN / Transformer，就一步步逼近真正的 GPT 了。
